In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# =========================
# =========================

data_dir = r"D:\은기\진행중\PFAS_classification\PFAS_classification_Re"
file = "Chemotype_rearranged.xlsx"
label_path = os.path.join(data_dir, "label.xlsx")

df_raw = pd.read_excel(os.path.join(data_dir, file), index_col=0).T
label_df = pd.read_excel(label_path, index_col=0)

df_raw["Label"] = label_df.iloc[:, 0].loc[df_raw.index]

X_raw = df_raw.drop(columns=["Label"])
y_raw = df_raw["Label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

original_columns = X_raw.columns.tolist()
feature_name_map = {
    f"F{str(i+1).zfill(3)}": col
    for i, col in enumerate(original_columns)
}

X_raw = X_raw.copy()
X_raw.columns = list(feature_name_map.keys())

X_full = X_raw.copy()
y = pd.Series(y_encoded, index=X_full.index)


# =========================
# 🤖 모델 정의
# =========================

model_dict = {
    "LR": LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),

    "SVM": LinearSVC(
        C=0.5,
        penalty="l1",
        dual=False,
        random_state=42,
        max_iter=5000
    ),

    "MLP": MLPClassifier(
        random_state=42,
        max_iter=1000
    ),

    "RF": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
     n_estimators=100,
     learning_rate=0.1,
     max_depth=3,
     eval_metric="mlogloss",
     random_state=42
    )
}


# =========================
# =========================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

max_features = 15


# =========================
# =========================

def get_feature_ranking(model, X_train, y_train):
    """
    Feature ranking is calculated using training data only.
    """

    model_clone = clone(model)
    model_clone.fit(X_train, y_train)

    if hasattr(model_clone, "coef_"):
        importances = np.mean(np.abs(model_clone.coef_), axis=0)

    elif hasattr(model_clone, "feature_importances_"):
        importances = model_clone.feature_importances_

    else:
        result = permutation_importance(
            model_clone,
            X_train,
            y_train,
            scoring="accuracy",
            n_repeats=5,
            random_state=42
        )
        importances = result.importances_mean

    feat_series = pd.Series(importances, index=X_train.columns)
    sorted_features = feat_series.sort_values(ascending=False).index.tolist()

    return sorted_features


# =========================
# =========================

best_feature_summary = {}
feature_performance_by_model = {}
best_df_dict = {}
summary_stats = []

for model_name, model in model_dict.items():

    print(f"\n🔍 Evaluating model: {model_name}")

    outer_acc_scores = []
    outer_f1_scores = []
    outer_predictions = pd.Series(index=X_full.index, dtype=int)

    selected_features_all_folds = []
    selected_k_all_folds = []
    confusion_matrices = []

    for outer_fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_full, y),
        start=1
    ):

        X_train_outer = X_full.iloc[train_idx]
        y_train_outer = y.iloc[train_idx]

        X_test_outer = X_full.iloc[test_idx]
        y_test_outer = y.iloc[test_idx]

        # =========================
        # =========================

        k_scores = {}

        for k in range(1, min(max_features, X_full.shape[1]) + 1):

            inner_scores = []

            for inner_train_idx, inner_val_idx in inner_cv.split(
                X_train_outer,
                y_train_outer
            ):

                X_train_inner = X_train_outer.iloc[inner_train_idx]
                y_train_inner = y_train_outer.iloc[inner_train_idx]

                X_val_inner = X_train_outer.iloc[inner_val_idx]
                y_val_inner = y_train_outer.iloc[inner_val_idx]

                sorted_features_inner = get_feature_ranking(
                    model,
                    X_train_inner,
                    y_train_inner
                )

                selected_feats_inner = sorted_features_inner[:k]

                inner_model = clone(model)
                inner_model.fit(
                    X_train_inner[selected_feats_inner],
                    y_train_inner
                )

                y_val_pred = inner_model.predict(
                    X_val_inner[selected_feats_inner]
                )

                inner_acc = accuracy_score(y_val_inner, y_val_pred)
                inner_scores.append(inner_acc)

            k_scores[k] = np.mean(inner_scores)

        best_k = max(k_scores, key=k_scores.get)

        # =========================
        # =========================

        sorted_features_outer = get_feature_ranking(
            model,
            X_train_outer,
            y_train_outer
        )

        selected_feats_outer = sorted_features_outer[:best_k]

        final_model = clone(model)
        final_model.fit(
            X_train_outer[selected_feats_outer],
            y_train_outer
        )

        y_test_pred = final_model.predict(
            X_test_outer[selected_feats_outer]
        )

        outer_acc = accuracy_score(y_test_outer, y_test_pred)
        outer_f1 = f1_score(
            y_test_outer,
            y_test_pred,
            average="macro",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test_outer,
            y_test_pred,
            labels=np.arange(len(le.classes_))
        )

        outer_acc_scores.append(outer_acc)
        outer_f1_scores.append(outer_f1)
        confusion_matrices.append(cm)

        outer_predictions.iloc[test_idx] = y_test_pred

        selected_features_all_folds.append(selected_feats_outer)
        selected_k_all_folds.append(best_k)

        print(
            f"Fold {outer_fold}: "
            f"best_k={best_k}, "
            f"Accuracy={outer_acc:.4f}, "
            f"Macro F1={outer_f1:.4f}"
        )

    # =========================
    # =========================

    outer_acc_scores = np.array(outer_acc_scores)
    outer_f1_scores = np.array(outer_f1_scores)

    failed_samples = X_full.index[outer_predictions != y].tolist()

    flattened_features = [
        feat
        for fold_feats in selected_features_all_folds
        for feat in fold_feats
    ]

    feature_freq = pd.Series(flattened_features).value_counts()

    frequent_features_original_names = [
        feature_name_map.get(f, f)
        for f in feature_freq.index.tolist()
    ]

    total_confusion_matrix = np.sum(confusion_matrices, axis=0)

    best_feature_summary[model_name] = {
        "num_features_each_fold": selected_k_all_folds,
        "mean_num_features": round(np.mean(selected_k_all_folds), 2),
        "feature_names_by_frequency": frequent_features_original_names,
        "feature_frequency": {
            feature_name_map.get(k, k): int(v)
            for k, v in feature_freq.items()
        },
        "accuracy_mean": round(outer_acc_scores.mean(), 4),
        "accuracy_std": round(outer_acc_scores.std(), 4),
        "macro_f1_mean": round(outer_f1_scores.mean(), 4),
        "macro_f1_std": round(outer_f1_scores.std(), 4),
        "failed_samples": failed_samples,
        "confusion_matrix": total_confusion_matrix
    }

    feature_performance_by_model[model_name] = {
        "outer_accuracy_scores": outer_acc_scores,
        "outer_macro_f1_scores": outer_f1_scores,
        "selected_k_all_folds": selected_k_all_folds,
        "selected_features_all_folds": selected_features_all_folds,
        "confusion_matrices": confusion_matrices,
        "total_confusion_matrix": total_confusion_matrix
    }

    summary_stats.append({
        "Model": model_name,
        "Accuracy Mean": round(outer_acc_scores.mean(), 4),
        "Accuracy Std": round(outer_acc_scores.std(), 4),
        "Macro F1 Mean": round(outer_f1_scores.mean(), 4),
        "Macro F1 Std": round(outer_f1_scores.std(), 4),
        "Mean Selected Features": round(np.mean(selected_k_all_folds), 2),
        "Selected Features per Fold": selected_k_all_folds
    })

    print(f"\n⭐ {model_name}")
    print(f"📈 Accuracy: {outer_acc_scores.mean():.4f} ± {outer_acc_scores.std():.4f}")
    print(f"📊 Macro F1: {outer_f1_scores.mean():.4f} ± {outer_f1_scores.std():.4f}")
    print(f"🔢 Selected k per fold: {selected_k_all_folds}")
    print("🧬 Frequently selected features:")
    print(frequent_features_original_names[:20])
    print(f"❌ Failed samples: {failed_samples if failed_samples else 'None'}")
    print("🧩 Total confusion matrix:")
    print(total_confusion_matrix)


🔍 Evaluating model: LR
Fold 1: best_k=6, Accuracy=0.3333, Macro F1=0.2222
Fold 2: best_k=14, Accuracy=0.0000, Macro F1=0.0000
Fold 3: best_k=1, Accuracy=0.6667, Macro F1=0.5556
Fold 4: best_k=1, Accuracy=0.3333, Macro F1=0.1667
Fold 5: best_k=1, Accuracy=0.0000, Macro F1=0.0000

⭐ LR
📈 Accuracy: 0.2667 ± 0.2494
📊 Macro F1: 0.1889 ± 0.2037
🔢 Selected k per fold: [6, 14, 1, 1, 1]
🧬 Frequently selected features:
['bond:COH_alcohol_pri-alkyl', 'bond:N[!C]_amino', 'bond:COH_alcohol_generic', 'bond:COH_alcohol_aliphatic_generic', 'bond:C(~Z)~C~Q_a-haloether', 'bond:COH_alcohol_diol_(1_1-)', 'bond:CX_halide_alkyl-F_perfluoro_butyl', 'bond:CX_halide_alkyl-F_tetrafluoro_(1_1_1_2-)', 'bond:S=O_sulfonyl_generic', 'bond:CX_halide_alkyl-X_trihalo_(1_2_3-)', 'bond:CX_halide_alkyl-X_trihalo_(1_1_1-)', 'bond:COC_ether_aliphatic', 'bond:CX_halide_alkyl-F_trifluoro_(1_1_1-)', 'chain:alkaneLinear_ethyl_C2(H_gt_1)', 'atom:element_metal_group_I_II', 'bond:CX_halide_alkyl-F_perfluoro_hexyl', 'bond:CX_halid

In [5]:
# =========================
# =========================

summary_df = pd.DataFrame(summary_stats)
display(summary_df)

for model_name in model_dict.keys():
    cm = feature_performance_by_model[model_name]["total_confusion_matrix"]

    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in le.classes_],
        columns=[f"Pred_{c}" for c in le.classes_]
    )

    print(f"\nConfusion Matrix - {model_name}")
    display(cm_df)

,Model,Accuracy Mean,Accuracy Std,Macro F1 Mean,Macro F1 Std,Mean Selected Features,Selected Features per Fold
0,LR,0.2667,0.2494,0.1889,0.2037,4.6,"[6, 14, 1, 1, 1]"
1,SVM,0.3333,0.0000,0.2444,0.0754,4.6,"[5, 1, 10, 6, 1]"
2,MLP,0.2667,0.3266,0.2222,0.2722,6.6,"[1, 11, 5, 11, 5]"
3,RF,0.0667,0.1333,0.0444,0.0889,5.2,"[6, 14, 1, 3, 2]"
4,XGBoost,0.4667,0.1633,0.3333,0.1826,1.2,"[1, 1, 1, 1, 2]"



Confusion Matrix - LR


,Pred_1,Pred_2,Pred_3
True_1,2,2,1
True_2,2,2,1
True_3,2,3,0



Confusion Matrix - SVM


,Pred_1,Pred_2,Pred_3
True_1,3,0,2
True_2,3,2,0
True_3,4,1,0


...

,Pred_1,Pred_2,Pred_3
True_1,2,1,2
True_2,2,1,2
True_3,4,0,1



Confusion Matrix - RF


,Pred_1,Pred_2,Pred_3
True_1,0,2,3
True_2,1,1,3
True_3,2,3,0



Confusion Matrix - XGBoost


,Pred_1,Pred_2,Pred_3
True_1,4,1,0
True_2,2,3,0
True_3,3,2,0


In [6]:
import numpy as np
import pandas as pd

available_models = [
    m for m in model_dict.keys()
    if m in feature_performance_by_model
    and "total_confusion_matrix" in feature_performance_by_model[m]
]

print("Available models:", available_models)

summary_stats = []

for model_name in available_models:

    cm = feature_performance_by_model[model_name]["total_confusion_matrix"]

    TP = np.diag(cm)
    FP = np.sum(cm, axis=0) - TP
    FN = np.sum(cm, axis=1) - TP

    accuracy = np.trace(cm) / np.sum(cm)

    precision_per_class = np.divide(
        TP, TP + FP,
        out=np.zeros_like(TP, dtype=float),
        where=(TP + FP) != 0
    )

    sensitivity_per_class = np.divide(
        TP, TP + FN,
        out=np.zeros_like(TP, dtype=float),
        where=(TP + FN) != 0
    )

    f1_per_class = np.divide(
        2 * precision_per_class * sensitivity_per_class,
        precision_per_class + sensitivity_per_class,
        out=np.zeros_like(TP, dtype=float),
        where=(precision_per_class + sensitivity_per_class) != 0
    )

    summary_stats.append({
        "Model": model_name,
        "Accuracy": round(accuracy, 4),
        "Macro Precision": round(np.mean(precision_per_class), 4),
        "Macro Sensitivity": round(np.mean(sensitivity_per_class), 4),
        "Macro F1": round(np.mean(f1_per_class), 4),
        "Class-wise Precision": [round(x, 4) for x in precision_per_class],
        "Class-wise Sensitivity": [round(x, 4) for x in sensitivity_per_class]
    })

summary_df = pd.DataFrame(summary_stats)
display(summary_df)

Available models: ['LR', 'SVM', 'MLP', 'RF', 'XGBoost']


,Model,Accuracy,Macro Precision,Macro Sensitivity,Macro F1,Class-wise Precision,Class-wise Sensitivity
0,LR,0.2667,0.2063,0.2667,0.2323,"[0.3333, 0.2857, 0.0]","[0.4, 0.4, 0.0]"
1,SVM,0.3333,0.3222,0.3333,0.3000,"[0.3, 0.6667, 0.0]","[0.6, 0.4, 0.0]"
2,MLP,0.2667,0.3167,0.2667,0.2645,"[0.25, 0.5, 0.2]","[0.4, 0.2, 0.2]"
3,RF,0.0667,0.0556,0.0667,0.0606,"[0.0, 0.1667, 0.0]","[0.0, 0.2, 0.0]"
4,XGBoost,0.4667,0.3148,0.4667,0.3723,"[0.4444, 0.5, 0.0]","[0.8, 0.6, 0.0]"
